# K-means|| distribuito su Dask — Analisi

Notebook di orchestrazione: avvia il cluster, carica il dataset, esegue i test (singola run o benchmark su più combinazioni) e salva i risultati in `./results`.

Moduli usati:
- `kmeans_parallel.py` — algoritmo k-means|| distribuito
- `launch_cluster.py` — avvio/spegnimento del cluster Dask via SSH
- `data_loader.py` — caricamento del dataset
- `benchmark.py` — esecuzione dei test e salvataggio risultati

## 1. Import

In [7]:
import numpy as np
import pandas as pd
from src.kmeans_parallel import kmeans_parallel
from src.launch_cluster import launch_cluster, shutdown_cluster
from src.data_loader import load_dataset
from src.benchmark import run_single_test, run_benchmark, calculate_inertia

import time

## 2. Parametri configurabili

Modifica qui i valori per cambiare numero di worker, `k`, `l` (oversampling factor) e `r` (numero di round dell'inizializzazione parallela).

In [9]:
# --- Cluster ---
N_WORKERS = 8      # tra 1 e 8 (nodi disponibili in launch_cluster.py)
NUM_PARTITIONS = 8 * N_WORKERS   # regola empirica: >= n_threads_per_worker * n_workers

# --- Algoritmo k-means|| ---
K = 500                # numero di cluster finali
L = 250                # oversampling factor (assoluto). In alternativa: L = round(L_OVER_K * K)
R = 10                  # numero di round dell'inizializzazione parallela
MAX_ITER_FIT = 100      # iterazioni massime della fase di Lloyd's (fit)
SEED = 42

In [10]:
# --- Dataset ---
DATASET_URL_10PC = "https://ndownloader.figshare.com/files/5976042"
# link used by sklearn function fetch_kddcup99 (original link gives 403 error)
#***for 10% dataset***

DATASET_URL_FULL="https://ndownloader.figshare.com/files/5976045"
#***FULL DATASET***

RAW_GZ_PATH   = "/home/ubuntu/Project/libero_development/data/kddcup_data.gz" # compressed (.gz) dataset file
PARQUET_PATH = '/tmp/kddcup_data.parquet' # single Parquet file (ie a compressed format file but partition-able, unlike .gz) on master

# column names of KDD dataset, from source code of the above sklearn function;
# "protocol_type","service","flag" are non-numeric so they will be dropped later,
# as will be "label" and the constant column 'num_outbound_cmds' 
# (10% dataset might have more constant columns such as 'is_host_login')
COL_NAMES = [
    "duration","protocol_type","service","flag","src_bytes",
    "dst_bytes","land","wrong_fragment","urgent","hot",
    "num_failed_logins","logged_in","num_compromised","root_shell",
    "su_attempted","num_root","num_file_creations","num_shells",
    "num_access_files","num_outbound_cmds","is_host_login",
    "is_guest_login","count","srv_count","serror_rate",
    "srv_serror_rate","rerror_rate","srv_rerror_rate","same_srv_rate",
    "diff_srv_rate","srv_diff_host_rate","dst_host_count",
    "dst_host_srv_count","dst_host_same_srv_rate",
    "dst_host_diff_srv_rate","dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate","dst_host_serror_rate",
    "dst_host_srv_serror_rate","dst_host_rerror_rate",
    "dst_host_srv_rerror_rate","label"
]

## 3. Avvio del cluster

In [13]:
# DO NOT RUN if already existing!
cluster, client = launch_cluster(N_WORKERS)

Inizializzazione del cluster SSH con 8 worker...
Worker selezionati: ['10.67.22.254', '10.67.22.34', '10.67.22.145', '10.67.22.121', '10.67.22.192', '10.67.22.18', '10.67.22.187', '10.67.22.48']


2026-08-24 21:02:13,285 - distributed.deploy.ssh - INFO - 2026-08-24 21:02:13,284 - distributed.http.proxy - INFO - To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
2026-08-24 21:02:13,315 - distributed.deploy.ssh - INFO - 2026-08-24 21:02:13,314 - distributed.scheduler - INFO - State start
2026-08-24 21:02:13,320 - distributed.deploy.ssh - INFO - 2026-08-24 21:02:13,319 - distributed.scheduler - INFO -   Scheduler at:   tcp://10.67.22.194:8786
2026-08-24 21:02:14,903 - distributed.deploy.ssh - INFO - 2026-08-24 21:02:14,903 - distributed.nanny - INFO -         Start Nanny at: 'tcp://10.67.22.145:44971'
2026-08-24 21:02:14,916 - distributed.deploy.ssh - INFO - 2026-08-24 21:02:14,916 - distributed.nanny - INFO -         Start Nanny at: 'tcp://10.67.22.121:41283'
2026-08-24 21:02:14,918 - distributed.deploy.ssh - INFO - 2026-08-24 21:02:14,913 - distributed.nanny - INFO -         Start Nanny at: 'tcp://10.67.22.25

Cluster avviato e connessione stabilita con successo!



#### IF INSTEAD ALREADY EXISTING CLUSTER:

In [ ]:
from dask.distributed import Client

SCHEDULER_ADDRESS = "tcp://10.67.22.194:8786"

try:
    client = Client(SCHEDULER_ADDRESS, timeout="10s")
    print("Connected to cluster successfully!")
    print(f"Dask Dashboard link: {client.dashboard_link}")

except Exception as e:
    print(f"Connection error: {e}")

In [ ]:
client.scheduler_info()

## 4. Load dataset

In [14]:
start=time.time()
X_bag, (mean_ar, std_ar)=load_dataset(n_partitions=NUM_PARTITIONS,
                   client=client,
                   dataset_url=DATASET_URL_10PC,
                   raw_gz_path=RAW_GZ_PATH,
                   parquet_path=PARQUET_PATH,
                   parquet_path_workers=PARQUET_PATH,
                   col_names=COL_NAMES)
end=time.time()
elapsed=end-start
print(f"Time elapsed: {elapsed:.2f} s")

Using cached dataset: /home/ubuntu/Project/libero_development/data/kddcup_data.gz
Converting .gz -> Parquet chunk sizes...
Parquet file created (compressed with snappy).
Number of partitions before preprocessing: 64
Constant columns: ['num_outbound_cmds', 'is_host_login']
Computing global mean and std (first pass over data)...
Distributed bag created with 32 partitions.
Number of samples: 494021
Time elapsed: 11.50 s


In [ ]:
# if want to go back to original coordinates later
#print(mean_ar, '\n\n\n') 
#print(std_ar)

#### Check that loaded correctly

In [ ]:
# global variable used in data_loader.py 
print(f"Number of initial features (should be 42): {len(COL_NAMES)}")

In [ ]:
lengths = X_bag.map(len).frequencies().compute() # length of every row- its unnormalized PMF
print("Feature count distribution (should be 37 for full dataset, see below):", lengths)
# Should show one unique length, 37 (see below)

In [ ]:
n_points = X_bag.count().compute()
n_features = X_bag.map(len).take(1)[0] # since all rows have same length
print(f"Points: {n_points}, Features: {n_features}\n(Full dataset should contain 4898431 samples with 37 (=42 -1 label -3 non numeric -1 constant feature) numerical, non-constant features)")

In [ ]:
has_nan = X_bag.map(lambda x: np.isnan(x).any()).any().compute()
has_inf = X_bag.map(lambda x: np.isinf(x).any()).any().compute()
print(f"Has NaN: {has_nan}, Has Inf: {has_inf}\n(Both must be False)")

In [ ]:
total = X_bag.reduction(
    lambda part: np.sum(np.vstack(list(part)), axis=0),
    lambda parts: np.sum(list(parts), axis=0)
).compute()
mean = total / n_points
means_vector=np.any(np.abs(mean)>1e-3)
print(f"Any column mean significantly different from zero: {means_vector==True}\n(Must be False)")

In [ ]:
sq_total = X_bag.reduction(
    lambda part: np.sum(np.vstack(list(part))**2, axis=0),
    lambda parts: np.sum(list(parts), axis=0)
).compute()

std = np.sqrt(sq_total / n_points - mean**2)
std_minus_one_vector=np.abs(std - 1)
print(f"Any column where std significantly different from one: {np.any(std_minus_one_vector > 1e-3)}\n(Must be False)")

In [ ]:
sample_rows = X_bag.take(5)
for i, row in enumerate(sample_rows):
    print(f"Row {i}: shape={row.shape}, first 5 values={row[:5]}")

In [ ]:
# additional sanity check
print("Type of X_bag:", type(X_bag))
print("Number of partitions:", X_bag.npartitions)

## 5. Single run

Runs a single combination of parameters (the ones defined in section 2) and prints the cost and execution time.

In [ ]:
start=time.time()
result, _ = run_single_test(
    client,
    k=K, l=L, r=R,
    num_partitions=NUM_PARTITIONS,
    max_iter_fit=MAX_ITER_FIT,
    seed=SEED,
    X_bag=X_bag,
    track_convergence=True,
    track_centroids=True
)
result
end=time.time()
elapsed=end-start
print(f"Time elapsed: {elapsed:.2f} s")

In [ ]:
#cost =567232.43 
#cost_ten=cost * 1e-10
#print(f"(Cost equivalent to {cost_ten :.2f} * 10^10)")
minutes=elapsed/60
print(f"(Time elapsed equivalent to {minutes:.2f} minutes)")

In [ ]:
import matplotlib.pyplot as plt

iterations = range(1, len(result["cost_history"]) + 1)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(iterations, result["cost_history"], marker="o", markersize=4, linewidth=2, color="tab:blue")
ax.set_xlabel("Iteration")
ax.set_ylabel("Cost (inertia)")
ax.set_title(f"K-means convergence (k={result['k']})")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
centroids_table = pd.DataFrame({
    "round": range(1, len(result["n_centroids_history"]) + 1),
    "n_centroids": result["n_centroids_history"],
})
centroids_table.style.set_caption(f"l = {result['l']}")

Compare with:
* Cost $19 \cdot 10^{10}$ ($k=500$, $l=0.5 k$, but $r=5$ instead - table 3.).
  HOWEVER, note that here we standardized data beforehand!
* Running time $69.0 \text{ min}$ for the same values (table 4)

## 6. Benchmark multi-combinazione

Testa più combinazioni di `(n_workers, partitions, l_over_k, r)` per diversi valori di `k`, e salva tutto in `./results`. 

**Nota:** il numero di worker effettivamente attivi nel cluster è quello impostato con `launch_cluster` in sezione 3 — la colonna `workers` qui sotto serve solo per etichettare/loggare i risultati, non riavvia il cluster.

In [ ]:
combinations = [
    # (n_workers, num_partitions, l_over_k, r)
    (N_WORKERS, NUM_PARTITIONS // 2, 1, R),   # under-partitioned: fewer partitions than total threads
    (N_WORKERS, NUM_PARTITIONS, 1, R),        # balanced: 1 partition per thread
    (N_WORKERS, NUM_PARTITIONS + 1, 1, R),    # imbalanced: 1 thread gets 2 partitions
    (N_WORKERS, NUM_PARTITIONS * 2, 1, R),    # over-partitioned: 2 partitions per thread
]

K_VALUES = [1000]

df_results = run_benchmark(
    client, X_bag,
    combinations=combinations,
    k_values=K_VALUES,
    label="kddcup99_benchmark",
    max_iter_fit=MAX_ITER_FIT,
    seed=SEED,
    averaging_iterations = 10
)
df_results

## 6.1 Analisi dati

Per vedere i risultati ottenuti precedentemente

In [ ]:
from src.benchmark_analysis import BenchmarkAnalyzer

analyzer = BenchmarkAnalyzer(
    data_path="/home/ubuntu/Project/working/results/kddcup99_benchmark_20260714_205025.csv",
    output_dir="figures",
    facet_cols=["k"],              # una figura per ogni valore (combinazione) di queste colonne
    x_col="partitions",               # variabile sull'asse x
    metrics=["cost", "time"],       # colonne di cui calcolare mean/std e plottare
)
grouped = analyzer.compute_grouped_stats(groupby_cols=["k", "partitions"])
analyzer.print_summary(grouped)
analyzer.plot_all(grouped)

## 7. Spegnimento del cluster

Da eseguire a fine lavoro, o prima di rilanciare `launch_cluster` con un `N_WORKERS` diverso.

In [ ]:
shutdown_cluster(cluster, client)